import all of the dependencies for importing

In [ ]:
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
from Foundation import NSData,  NSUnarchiver
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import os
import torch
import json
from dotenv import load_dotenv
from dataclasses import dataclass
from collections import defaultdict
import sqlite3

load_dotenv(dotenv_path=os.path.join(os.path.dirname(__file__), "..", ".env"))


set the model parameters and the LoRA parameters

In [ ]:
@dataclass
class model_params:
    model_id: str = "Qwen/Qwen3-30B-A3B"
    tokenizer_id: str="Qwen/Qwen3-30B-A3B"
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: int = 0.1

cfg = model_params()

create the dataset

In [ ]:
db_path = "new_chat.db"  # <-- change this line only

conn = sqlite3.connect(db_path)
cur = conn.cursor()

# 1️⃣ One-to-one chat IDs
cur.execute("""
    SELECT c.ROWID
    FROM chat c
    JOIN chat_handle_join chj ON chj.chat_id = c.ROWID
    GROUP BY c.ROWID
    HAVING COUNT(DISTINCT chj.handle_id) = 1
""")
chat_ids = [row[0] for row in cur.fetchall()]
print(f"Found {len(chat_ids)} 1:1 chats")

if not chat_ids:
    print("No eligible chats found.")
else:
    # 2️⃣ Fetch clean messages
    placeholder = ",".join("?" for _ in chat_ids)
    query = f"""
        SELECT cmj.chat_id, m.text, m.is_from_me, m.date,
               m.associated_message_guid, m.message_action_type,
               m.item_type, m.is_system_message, m.is_service_message,
               m.is_corrupt, m.thread_originator_guid, m.thread_originator_part
        FROM message m
        JOIN chat_message_join cmj ON cmj.message_id = m.ROWID
        WHERE cmj.chat_id IN ({placeholder})
        ORDER BY cmj.chat_id, m.date, m.ROWID
    """
    cur.execute(query, chat_ids)

    messages = []
    for row in cur.fetchall():
        (chat_id, text, is_from_me, date,
         assoc_guid, action_type, item_type,
         is_sys, is_service, is_corrupt,
         thread_guid, thread_part) = row

        if not text or not str(text).strip():
            continue
        if any([is_sys, is_service, is_corrupt, assoc_guid, thread_guid, thread_part]):
            continue
        if action_type not in (None, 0) or item_type not in (None, 0):
            continue

        messages.append((chat_id, text.strip(), is_from_me, date))

    # 3️⃣ Pair incoming → outgoing
    by_chat = defaultdict(list)
    for msg in messages:
        by_chat[msg[0]].append(msg)

    pairs = []
    for chat_id, msgs in by_chat.items():
        for i in range(len(msgs) - 1):
            curr, nxt = msgs[i], msgs[i + 1]
            if curr[2] == 0 and nxt[2] == 1:  # incoming → outgoing
                pairs.append((chat_id, curr[1], nxt[1]))

    print(f"Found {len(pairs)} message pairs")

    # 4️⃣ Show examples
    for i, (chat_id, insert_text, response_text) in enumerate(pairs[:5]):
        print(f"\n[Pair {i+1}] Chat ID: {chat_id}")
        print(f"Incoming: {insert_text}")
        print(f"Response: {response_text}")

conn.close()